# Titanic — ניתוח נתונים

ניתוח תיאורי של `Titanic.csv` (891 נוסעים). היעד: **Survived** (1 = שרד).
ללא מודל חיזוי.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("Titanic.csv")
print(df.shape)
df.head()

## סקירה וערכים חסרים

In [ ]:
df.info()
display(df.describe(include="all").T)
missing = df.isna().sum().to_frame("חסר")
missing["אחוז"] = (missing["חסר"] / len(df) * 100).round(1)
missing[missing["חסר"] > 0]

## הישרדות כללית, לפי מין ומחלקה

In [ ]:
print("שיעור הישרדות כללי:", round(df["Survived"].mean(), 3))
print("שרדו / לא:", df["Survived"].value_counts().to_dict())

sex = df.groupby("Sex")["Survived"].agg(["count", "sum", "mean"])
pclass = df.groupby("Pclass")["Survived"].agg(["count", "sum", "mean"])
sex_class = df.pivot_table("Survived", index="Sex", columns="Pclass", aggfunc="mean")

display(sex, pclass, sex_class.round(3))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
df["Survived"].value_counts().sort_index().plot.bar(ax=axes[0], title="Survived")
sex["mean"].plot.bar(ax=axes[1], title="הישרדות לפי מין")
pclass["mean"].plot.bar(ax=axes[2], title="הישרדות לפי מחלקה")
for ax in axes:
    ax.set_ylabel("שיעור / מספר")
plt.tight_layout()
plt.show()

## גיל, משפחה, מחיר ותא

In [ ]:
known_age = df.dropna(subset=["Age"]).copy()
known_age["AgeGroup"] = pd.cut(
    known_age["Age"],
    bins=[0, 12, 18, 35, 50, 80],
    labels=["0-12", "13-18", "19-35", "36-50", "51+"],
)

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["HasCabin"] = df["Cabin"].notna()

display(known_age.groupby("AgeGroup", observed=True)["Survived"].agg(["count", "mean"]))
display(df.groupby(df["FamilySize"].eq(1).map({True: "לבד", False: "עם משפחה"}))["Survived"].mean())
display(df.groupby("FamilySize")["Survived"].agg(["count", "mean"]))
display(df.groupby("Survived")["Fare"].agg(["mean", "median"]))
display(df.groupby("HasCabin")["Survived"].mean())

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
known_age.boxplot(column="Age", by="Survived", ax=axes[0])
axes[0].set_title("גיל לפי הישרדות")
axes[0].get_figure().suptitle("")
known_age.groupby("AgeGroup", observed=True)["Survived"].mean().plot.bar(ax=axes[1], title="הישרדות לפי גיל")
df.groupby("FamilySize")["Survived"].mean().plot.bar(ax=axes[2], title="הישרדות לפי גודל משפחה")
plt.tight_layout()
plt.show()

## נמל עלייה ותואר בשם

In [ ]:
df["Title"] = df["Name"].str.extract(r",\s*([A-Za-z]+)\.")
title_keep = ["Mr", "Mrs", "Miss", "Master"]
df["TitleGroup"] = df["Title"].where(df["Title"].isin(title_keep), "Other")

display(df.groupby("Embarked")["Survived"].agg(["count", "mean"]))
display(df.groupby("TitleGroup")["Survived"].agg(["count", "mean"]).sort_values("mean", ascending=False))

## מסקנות קצרות

1. **מין ומחלקה** מסבירים את רוב הפער: נשים במחלקה 1–2 כמעט תמיד שרדו; גברים במחלקה 3 כמעט תמיד לא.
2. **ילדים** ונוסעים במשפחה קטנה–בינונית שרדו יותר מנוסעים לבד; משפחות גדולות מאוד — שיעור נמוך.
3. **מחיר כרטיס ומספר תא** בעיקר פרוקסי למעמד (מחלקה 1).
4. ערכים חסרים: `Cabin` ~77%, `Age` ~20%, `Embarked` 2 שורות.